In [ ]:
!pip install pyannote.core pyannote.metrics resemblyzer spectralcluster librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 6.6 MB/

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/spktsagar/wav2vec2-large-xls-r-300m-nepali-openslr

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/spktsagar/wav2vec2-large-xls-r-300m-nepali-openslr)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="spktsagar/wav2vec2-large-xls-r-300m-nepali-openslr")

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained("spktsagar/wav2vec2-large-xls-r-300m-nepali-openslr")
model = AutoModelForCTC.from_pretrained("spktsagar/wav2vec2-large-xls-r-300m-nepali-openslr")

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [ ]:
import librosa
import numpy as np
from resemblyzer import VoiceEncoder, preprocess_wav
from spectralcluster import SpectralClusterer
from pyannote.core import Annotation, Segment

# ------------ CONFIG ----------------
SAMPLE_RATE = 16000
WINDOW = 1.5     # window size in seconds
STEP = 0.75      # step size in seconds
# ------------------------------------

def extract_embeddings(audio_path):
    wav, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
    encoder = VoiceEncoder()

    segments = []
    embeddings = []

    for start in np.arange(0, len(wav) / sr - WINDOW, STEP):
        s = int(start * sr)
        e = int((start + WINDOW) * sr)
        segment_wav = wav[s:e]

        if len(segment_wav) < WINDOW * sr:
            continue

        emb = encoder.embed_utterance(segment_wav)
        embeddings.append(emb)
        segments.append((start, start + WINDOW))

    return np.array(embeddings), segments


def cluster_embeddings(embeddings, min_clusters=2, max_clusters=10):
    clusterer = SpectralClusterer(
        min_clusters=min_clusters,
        max_clusters=max_clusters,
        autotune=None
    )
    labels = clusterer.predict(embeddings)
    return labels


def build_rttm(segments, labels):
    ann = Annotation()

    for (start, end), spk in zip(segments, labels):
        ann[Segment(start, end)] = f"SPEAKER_{spk}"

    return ann


def diarize(audio_path, min_spk=2, max_spk=3):
    print("Extracting embeddings...")
    emb, seg = extract_embeddings(audio_path)

    print("Clustering speakers...")
    labels = cluster_embeddings(emb, min_clusters=min_spk, max_clusters=max_spk)

    print("Building RTTM...")
    ann = build_rttm(seg, labels)
    print("Diarization done.")
    return ann


In [ ]:
ann = diarize("conv_0.wav", min_spk=2, max_spk=3)
print(ann)

Extracting embeddings...
Loaded the voice encoder model on cuda in 0.17 seconds.
Clustering speakers...
Building RTTM...
Diarization done.
[ 00:00:00.000 -->  00:00:01.500] _ SPEAKER_0
[ 00:00:00.750 -->  00:00:02.250] _ SPEAKER_0
[ 00:00:01.500 -->  00:00:03.000] _ SPEAKER_0
[ 00:00:02.250 -->  00:00:03.750] _ SPEAKER_0
[ 00:00:03.000 -->  00:00:04.500] _ SPEAKER_0
[ 00:00:03.750 -->  00:00:05.250] _ SPEAKER_0
[ 00:00:04.500 -->  00:00:06.000] _ SPEAKER_0
[ 00:00:05.250 -->  00:00:06.750] _ SPEAKER_1
[ 00:00:06.000 -->  00:00:07.500] _ SPEAKER_1
[ 00:00:06.750 -->  00:00:08.250] _ SPEAKER_1


In [ ]:
import librosa
import torch


audio_path = "conv_0.wav"
y, sr = librosa.load(audio_path, sr=16000)

speaker_text = {}

for segment, track, label in ann.itertracks(yield_label=True):
    start = int(segment.start * sr)
    end = int(segment.end * sr)
    audio_seg = y[start:end]

    inputs = processor(audio_seg, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(inputs.input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(predicted_ids[0])

    if label not in speaker_text:
        speaker_text[label] = []

    speaker_text[label].append(transcription)

print(speaker_text)

{'SPEAKER_0': ['समुहत', 'मूह तथा व्यक्ति', 'व्यक्ति', '', 'दुई रूपैयाँ', 'पैयाँ दरक', 'दरको'], 'SPEAKER_1': ['यस', 'यस मन्त्रालयल', 'मन्त्रालयलाई कहिले']}
